In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langgraph.graph import START, END , StateGraph
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.types import interrupt , Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal

In [ ]:
llm = ChatGroq(model = "openai/gpt-oss-20b")

In [ ]:
class MailState(BaseModel):
    query: str = ""
    draft: str = ""
    human_feedback:str = ""
    final_response:str = ""

##### Defining the nodes

In [ ]:
def DraftEmail(state:MailState) -> MailState:
    print("Generating Draft--->")

    if state.human_feedback:
        draft = llm.invoke(f"""
                           Rewrite this draft email for {state.query},
                           You hv generated this mail: {state.draft},
                           Human Feedback to apply: {state.human_feedback}
                           """).content
    else:
        draft = llm.invoke(state.query).content

    state.draft = draft


    print("Draft Completed, now ask for feedback......")

    return state 


def HumanFeedback(state:MailState) -> MailState:
    print("Asking for Feedback........")
    feedback = interrupt({
        "draft_email": state.draft,
        "question": "Do you want to continue or re-write the mail"
    })

    print("Got the FeedBack.....",feedback)

    fb = (feedback or "").strip().lower()
    if fb in ("approved", "ok","yes"):
        print("Approved........")
        state.human_feedback = ""
        return state 
    else : 
        print("Update Draft.....")
        state.human_feedback = feedback
        return state 



def FinalizeNode(state:MailState) -> MailState:
    print("Email sent Successfully....")
    state.final_response = "Email Sent Successfully"
    return state



def ConditionRouting(state: MailState) -> Literal["draft", "final"]:
    if state.human_feedback:
        print("Go and draft again.....")
        return "draft"

    print("Finally Draft Completed Successfully......")
    return "final"


##### Build Graph

In [ ]:
graph = StateGraph(MailState)


## add node
graph.add_node("draft",DraftEmail)
graph.add_node("human_fb",HumanFeedback)
graph.add_node("final",FinalizeNode)


## add edges
graph.add_edge(START,"draft") 
graph.add_edge("draft","human_fb")
graph.add_conditional_edges("human_fb",ConditionRouting)
graph.add_edge("final",END)


graph = graph.compile(checkpointer= InMemorySaver())

####  Now ask query to agent .........

In [ ]:
##  Here we are not adding Command for interrupt

res = graph.invoke({"query": "I want to send a email for Medical Leave Request, "
                       "dates will be 10 oct to 15 oct, so create a simple mail"},
             {
                 "configurable":{"thread_id":"1"}
             })


res
print(res["draft"])
#####  Abhi hmm Interrupt pe rukk gye hai :: waiting for Feedback.........


In [ ]:
##  adding command for interrupt :: giving Feedback
res = graph.invoke(Command(resume="It is from 15 to 20 OCT"), {
                 "configurable":{"thread_id":"1"}
             })             ####  ab Interupt se resume kro 

res

In [ ]:
res = graph.invoke(Command(resume="ok"), {
                 "configurable":{"thread_id":"1"}
             })

res

In [ ]:
from IPython.display import Image   # to display image of graph
Image(graph.get_graph().draw_mermaid_png())